In [ ]:
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/apikeys.txt')

True

Orchestration(오케스트레이션)  
-> 하나의 질문을 하위 n개로 나눠서 각 질문에 대한 응답을 받고 그 응답을 다시 LLM에 보내 요약/종합해서 하나의 최종 응답으로 만듦

In [ ]:
import json
import asyncio

In [ ]:
from openai import AsyncOpenAI, OpenAI

sync_client = OpenAI()
client = AsyncOpenAI() # OpenAI API를 '비동기 방식'으로 사용하기 위한 클라이언트 생성


In [ ]:
# 프롬프트 받아서 OpenAI API 호출하고 LLM의 답변 문자열을 돌려주는 함수
# LLM 호출은 동기로 진행
def llm_call(prompt:str, model:str = 'gpt-4o-mini') -> str:
  messages = [] # 여기에 OpenAI API가 받은 {role, content} 딕셔너리 넣어야 함
  messages.append({'role':'user','content':prompt})
  chat_completion = sync_client.chat.completions.create(model=model,messages=messages)
  return chat_completion.choices[0].message.content

In [ ]:
async def llm_call_async(prompt:str, model:str = 'gpt-4o-mini') -> str:
  messages = []
  messages.append({'role':'user','content':prompt})
  chat_completion = await client.chat.completions.create(
      model=model,
      messages=messages)
  print(model, '완료')
  return chat_completion.choices[0].message.content

In [ ]:
async def run_llm_parallel(prompt_list):
  tasks = [llm_call_async(prompt) for prompt in prompt_list]
  responses = []
  for task in asyncio.as_completed(tasks):
    result = await task
    responses.append(result)
  return responses

In [ ]:
# user_query : 사용자 질문
user_query = 'AI는 미래 일자리에 어떤 영향을 미칠까?'

In [ ]:
# orchestrator_prompt : user_query를 LLM이 어떻게 생각/어떤 순서/어떤 규칙으로 처리할지 지시
orchestrator_prompt= f"""
다음 사용자 질문을 분석하고, 이를 3개의 관련된 하위 질문으로 분해하십시오:

다음 형식으로 응답을 제공하십시오:

{{
    "analysis": "사용자 질문에 대한 이해를 상세히 설명하고, 작성한 하위 질문들의 근거를 설명하십시오.",
    "subtasks": [
        {{
            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",
            "sub_question": "질문 1"
        }},
        {{
            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",
            "sub_question": "질문 2"
        }}
        // 필요에 따라 추가 하위 질문 포함
    ]
}}
최대 3개의 하위 질문을 생성하세요

사용자 질문: {user_query}"""

In [ ]:
orchestrator_prompt

'\n다음 사용자 질문을 분석하고, 이를 3개의 관련된 하위 질문으로 분해하십시오:\n\n다음 형식으로 응답을 제공하십시오:\n\n{\n    "analysis": "사용자 질문에 대한 이해를 상세히 설명하고, 작성한 하위 질문들의 근거를 설명하십시오.",\n    "subtasks": [\n        {\n            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",\n            "sub_question": "질문 1"\n        },\n        {\n            "description": "이 하위 질문의 초점과 의도를 설명하십시오.",\n            "sub_question": "질문 2"\n        }\n        // 필요에 따라 추가 하위 질문 포함\n    ]\n}\n최대 3개의 하위 질문을 생성하세요\n\n사용자 질문: AI는 미래 일자리에 어떤 영향을 미칠까?'

In [ ]:
# 위 orchestrator_prompt를 LLM에 넘기면 응답을 줌
orchestrator_response = llm_call(orchestrator_prompt, model='gpt-4o')

In [ ]:
# LLM의 응답 결과
orchestrator_response

'```json\n{\n    "analysis": "사용자 질문은 인공지능(AI)이 미래의 고용 시장에 미치는 영향을 이해하고자 하는 것이다. 이는 경제, 사회, 기술적 측면에서 AI의 발전이 일자리의 성격과 수량에 미칠 잠재적 변화를 탐구하려는 것이다. 이러한 질문은 기술 혁신에 따른 자동화, 새롭게 떠오르는 직업 분야, 그리고 재교육 및 기술 습득의 필요성 등을 고려할 수 있다. 하위 질문들은 이러한 다양한 측면을 구체적으로 조사하는 데 중점을 둔다.",\n    "subtasks": [\n        {\n            "description": "AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.",\n            "sub_question": "AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?"\n        },\n        {\n            "description": "AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.",\n            "sub_question": "AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?"\n        },\n        {\n            "description": "AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.",\n            "sub_question": "미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?"\n 

In [ ]:
# LLM 오케스트레이션에서 쓰이는 후처리(post-processing) 코드
# LLM 응답에서 ```json 같은 코드블록 표시를 제거하고 순수 json 문자열만 남기려는 작업
orchestrator_response.replace('```json', '').replace('```', '')

'\n{\n    "analysis": "사용자 질문은 인공지능(AI)이 미래의 고용 시장에 미치는 영향을 이해하고자 하는 것이다. 이는 경제, 사회, 기술적 측면에서 AI의 발전이 일자리의 성격과 수량에 미칠 잠재적 변화를 탐구하려는 것이다. 이러한 질문은 기술 혁신에 따른 자동화, 새롭게 떠오르는 직업 분야, 그리고 재교육 및 기술 습득의 필요성 등을 고려할 수 있다. 하위 질문들은 이러한 다양한 측면을 구체적으로 조사하는 데 중점을 둔다.",\n    "subtasks": [\n        {\n            "description": "AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.",\n            "sub_question": "AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?"\n        },\n        {\n            "description": "AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.",\n            "sub_question": "AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?"\n        },\n        {\n            "description": "AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.",\n            "sub_question": "미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?"\n        

In [ ]:
# json으로 load하는 이유 : 문자열(text)로 받은 LLM 응답을 파이썬이 이해하고 조작할 수 있는 자료구조(dict, list)로 바꾸기 위해서
response_json = json.loads(orchestrator_response.replace('```json', '').replace('```', ''))

In [ ]:
response_json

{'analysis': '사용자 질문은 인공지능(AI)이 미래의 고용 시장에 미치는 영향을 이해하고자 하는 것이다. 이는 경제, 사회, 기술적 측면에서 AI의 발전이 일자리의 성격과 수량에 미칠 잠재적 변화를 탐구하려는 것이다. 이러한 질문은 기술 혁신에 따른 자동화, 새롭게 떠오르는 직업 분야, 그리고 재교육 및 기술 습득의 필요성 등을 고려할 수 있다. 하위 질문들은 이러한 다양한 측면을 구체적으로 조사하는 데 중점을 둔다.',
 'subtasks': [{'description': 'AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.',
   'sub_question': 'AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?'},
  {'description': 'AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.',
   'sub_question': 'AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?'},
  {'description': 'AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.',
   'sub_question': '미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?'}]}

In [ ]:
# response_json에 'analysis' 키값이 있으면 그 값을 가져오고 없으면 none
analysis = response_json.get('analysis', "")

In [ ]:
analysis

'사용자 질문은 인공지능(AI)이 미래의 고용 시장에 미치는 영향을 이해하고자 하는 것이다. 이는 경제, 사회, 기술적 측면에서 AI의 발전이 일자리의 성격과 수량에 미칠 잠재적 변화를 탐구하려는 것이다. 이러한 질문은 기술 혁신에 따른 자동화, 새롭게 떠오르는 직업 분야, 그리고 재교육 및 기술 습득의 필요성 등을 고려할 수 있다. 하위 질문들은 이러한 다양한 측면을 구체적으로 조사하는 데 중점을 둔다.'

In [ ]:
# response_json 보면 subtasks는 리스트 형태로 되어 있어서 없으면 빈 리스트([]) 반환 하는 것임
# 사용자 질문에 대한 하위 질문 들어있음
sub_tasks = response_json.get('subtasks', [])

In [ ]:
sub_tasks

[{'description': 'AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.',
  'sub_question': 'AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?'},
 {'description': 'AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.',
  'sub_question': 'AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?'},
 {'description': 'AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.',
  'sub_question': '미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?'}]

In [ ]:
# user_query : 사용자 질문
# sub_question, description : sub_tasks(LLM이 하위 질문 만든거)에 있음
def get_worker_prompt(user_query, sub_question, description):
  return f"""
  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:
  원래 질문 : {user_query}
  하위 질문 : {sub_question}

  지침 : {description}

  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요."""

In [ ]:
worker_prompts = [get_worker_prompt(user_query, task["sub_question"], task['description']) for task in sub_tasks]

In [ ]:
worker_prompts

['\n  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:\n  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?\n  하위 질문 : AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?\n\n  지침 : AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.\n\n  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.',
 '\n  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:\n  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?\n  하위 질문 : AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?\n\n  지침 : AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.\n\n  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.',
 '\n  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:\n  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?\n  하위 질문 : 미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?\n\n  지침 : AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.\n\n  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.']

In [ ]:
for i in worker_prompts:
  print(i)


  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:
  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
  하위 질문 : AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?

  지침 : AI의 발전이 자동화와 직무 대체를 통해 기존 일자리의 소멸 또는 변화를 어떻게 초래할 수 있는지 조사한다. 이는 어떤 직무가 AI에 의해 자동화될 가능성이 높고, 그로 인해 영향을 받을 산업군이 무엇인지를 탐구하려는 의도가 있다.

  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.

  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:
  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
  하위 질문 : AI로 인해 새롭게 창출될 수 있는 직업이나 산업은 무엇일까?

  지침 : AI 기술의 발전이 새로운 직업 또는 산업을 창출할 가능성과 그로 인한 고용 기회를 조사한다. 이는 AI와 관련된 신생 기술 분야에서의 성장 가능성과 그러한 분야들에서 필요로 하는 기술과 지식을 이해하려는 의도가 있다.

  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.

  다음 사용자 질문에서 파생된 하위 질문을 다루는 작업을 맡았습니다.:
  원래 질문 : AI는 미래 일자리에 어떤 영향을 미칠까?
  하위 질문 : 미래의 일자리 변화에 대비하기 위해 어떤 기술이나 역량이 필요할까?

  지침 : AI로 인해 변화하는 일자리 환경에서 필요한 새로운 기술이나 역량에 대해 조사한다. 이는 현직자들이 자신들의 역량을 업그레이드하거나 전직을 위해 필요한 교육 및 훈련 전략에 대해 이해하려는 의도를 포함한다.

  하위 질문을 철저히 다루는 포괄적이고 상세한 응답을 해주세요.


In [ ]:
# worker_responses : sub_tasks(하위질문)에 대한 응답
worker_responses = await run_llm_parallel(worker_prompts)

gpt-4o-mini 완료
gpt-4o-mini 완료
gpt-4o-mini 완료


In [ ]:
print(worker_responses[0])

NameError: name 'worker_responses' is not defined

In [ ]:
aggregator_prompt = f"""아래는 사용자의 원래 질문에 대해서 하위 질문을 나누고 응답한 결과입니다.
아래 질문 및 응답내용을 포함한 최종 응답을 제공해주세요.
## 요청사항
하위질문 응답내용이 최대한 포괄적이고 상세하게 포함되어야 합니다
사용자의 원래 질문:
{user_query}

하위 질문 및 응답:
"""

In [ ]:
for i in range(len(sub_tasks)):
  aggregator_prompt += f"\n{i+1}, 하위 질문 : {sub_tasks[i]['sub_question']}\n"
  aggregator_prompt += f"\n     응답 : {worker_responses[i]}"

In [ ]:
print(aggregator_prompt)

아래는 사용자의 원래 질문에 대해서 하위 질문을 나누고 응답한 결과입니다.
아래 질문 및 응답내용을 포함한 최종 응답을 제공해주세요.
## 요청사항
하위질문 응답내용이 최대한 포괄적이고 상세하게 포함되어야 합니다
사용자의 원래 질문:
AI는 미래 일자리에 어떤 영향을 미칠까?

하위 질문 및 응답:

1, 하위 질문 : AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?

     응답 : AI의 발전은 미래의 일자리 환경에 심대한 영향을 미칠 것으로 예상되며, 이로 인해 다양한 기술과 역량의 필요성이 증가하고 있습니다. 현직자들이 변화하는 시장에 대비하고 경쟁력을 유지하기 위해 필요한 교육 및 훈련 전략을 살펴보겠습니다.

### 1. 기술이나 역량의 변화

#### a. 디지털 리터러시
- **이해의 필요성**: 기본적인 컴퓨터 사용 능력에 그치지 않고, 데이터 분석, 클라우드 컴퓨팅, 소셜 미디어 활용 등의 디지털 기술에 대한 이해도가 필수적입니다.
- **학습 방법**: 온라인 강의, 자율 학습 플랫폼, 코딩 부트캠프 등 다양한 방법으로 디지털 기술을 배울 수 있습니다.

#### b. 데이터 분석 능력
- **내용의 중요성**: AI는 데이터 분석의 중심에 있습니다. 데이터 수집 및 해석 능력은 중요한 경쟁력이 됩니다.
- **교육 전략**: 데이터 과학, 통계학, 머신러닝 등의 과정을 통해 기초부터 심화 과정까지 학습할 수 있습니다.

#### c. 문제 해결 및 비판적 사고
- **원하는 능력**: AI가 처리하기 어려운 복잡하고 비정형적인 문제를 창의적으로 해결할 능력이 중요합니다.
- **강화 방법**: 실제 사례를 통해 문제 해결 프로젝트 수행, 팀워크 경험을 통한 시나리오 기반 학습 등이 도움을 줄 수 있습니다.

#### d. 소프트 스킬
- **소통 및 협업**: AI와 협력하고, 결과를 공유하며 팀워크를 이루기 위해서는 강력한 커뮤니케이션 능력이 필수입니다.
- **강화 방안**: 소통 능력을 배양하는 워크숍, 팀 프로젝트 경

In [ ]:
final_response = llm_call(aggregator_prompt, model='gpt-5.2')
print(final_response)

## AI는 미래 일자리에 어떤 영향을 미칠까? (최종 종합 응답)

AI는 데이터 분석, 머신러닝, 자연어 처리, 로봇 자동화 등의 기술 발전을 바탕으로 **반복적·규칙적 업무를 빠르게 자동화**하는 동시에, **새로운 직업과 산업을 만들어내고** 기존 직업의 역할을 **재정의(업무 재구성)**할 가능성이 큽니다. 즉, “일자리의 소멸”과 “일자리의 창출”, 그리고 “직무 내용의 변화”가 동시에 진행될 것입니다. 아래는 하위 질문에서 다룬 내용을 포괄적으로 통합한 정리입니다.

---

## 1) AI는 현재 존재하는 어떤 일자리를 대체할 가능성이 높을까?

AI가 특히 잘하는 영역은 **대량의 데이터를 빠르게 처리**하고, **정해진 규칙에 따라 일관되게 판단**하며, **반복 작업을 오류 없이 수행**하는 것입니다. 따라서 아래와 같은 직무가 자동화(대체) 영향을 더 크게 받을 가능성이 높습니다.

### 1. 단순 반복 작업
- **제조업 생산/조립 라인**: 산업용 로봇과 비전 AI가 반복 공정을 더 빠르고 정확하게 수행하면서, 단순 조립·검수 인력 수요가 감소할 수 있습니다.
- **데이터 입력/정리 업무**: OCR(광학 문자 인식), 문서 이해 AI, 자동 분류 시스템이 확산되면서 입력·전산 처리 중심 업무는 자동화 가능성이 큽니다.

### 2. 고정된 규칙 기반 업무
- **회계·세무의 반복 처리 영역**: 청구서 처리, 비용 정산, 기본 재무보고 생성 등 정형 업무는 회계 소프트웨어/AI로 자동화가 진전되고 있습니다.
- **콜센터 상담원(1차 응대)**: 챗봇·음성인식 시스템이 단순 문의 응대, 거래 처리, FAQ 대응을 대체하는 흐름이 이미 진행 중입니다.

### 3. 운송 및 물류 관련 업무
- **택시/배송/물류 운송**: 자율주행차, 배송 로봇, 드론 기술이 발전하면서 운전·배송 중심 업무의 구조가 바뀔 수 있습니다(전면 대체는 규제·안전·인프라에 좌우되나 변화 압력은 큼).

### 4. 정보 분석의 일부 영역
- **시장 조사

### Evaluation(평가)
-> 오케스트레이션 파이프라인이 의도한대로 잘 동작했는지 검증하는 단계  

<평가 목록>
1. 입력 이해 평가
2. 분해 평가 -> 적절한 하위 task로 나눴는지
3. 실행 평가 -> 하위 질문에 대한 응답이 정확한지
4. 종합 평가 -> 여러 응답이 잘 합쳐졌는지
5. 최종 추력 평가 -> 질문에 정확히 답했는지

In [ ]:
evaluator_prompt = """
다음 요약을 평가하십시오:

## 평가기준
1. 핵심 내용 포함 여부
   - 원문의 핵심 개념과 논리적 흐름이 유지되어야 합니다.
   - 불필요한 세부 사항은 줄이되, 핵심 정보가 누락되면 감점 요인입니다.
   - 단어 선택이 다소 달라도, 주요 개념과 의미가 유지되면 PASS 가능합니다.
   - 원문의 중요 개념 15% 이상이 빠졌다면 FAIL입니다.

2. 정확성 & 의미 전달
   - 요약이 원문의 의미를 왜곡하지 않고 정확하게 전달해야 합니다.
   - 숫자, 인명, 날짜 등 객관적 정보가 틀리면 FAIL입니다.
   - 문장이 다르게 표현되었더라도 원문의 의미를 유지하면 PASS 가능합니다.
   - 논리적 비약이 크거나 잘못된 해석이 포함되면 FAIL입니다.

3. 간결성 및 가독성
   - 문장이 과하게 길거나 반복적이면 감점 요인입니다.
   - 직역체 표현은 가독성을 해치지 않으면 허용 가능하지만, 지나치면 FAIL입니다.
   - 일부 단어의 표현 방식이 달라도 자연스럽다면 PASS 가능합니다.
   - 문장이 지나치게 어색해서 독해가 어렵다면 FAIL입니다.

4. 문법 및 표현
   - 맞춤법, 띄어쓰기 오류가 5개 이상이면 FAIL입니다.
   - 사소한 문법 실수는 감점 요인이나, 의미 전달에 영향을 주면 FAIL입니다.
   - 문장이 비문이거나 문맥상 어색한 표현이 많으면 FAIL입니다.

## 평가결과 응답예시
- 모든 기준이 충족되었으면 "평가결과 = PASS"를 출력하세요.
- 수정이 필요한 경우, 구체적인 문제점을 지적하고 반드시 개선 방향을 제시하세요.
- 중대한 오류가 있다면 "평가결과 = FAIL"을 출력하고, 반드시 주요 문제점을 설명하세요.

요약 결과 :
    """

In [ ]:
input_article = """
오픈AI가 몇 주 안에 새로운 모델인 'GPT-4.5'를 출시하며 분산돼 있던 생성형 인공지능(AI) 모델을 통합키로 했다. 추론용 모델인 'o' 시리즈를 정리하고 비(非)추론 모델인 'GPT' 시리즈로 합칠 예정이다.

13일 업계에 따르면 샘 알트먼 오픈AI 최고경영자(CEO)는 지난 12일 자신의 X(옛 트위터)에 'GPT-4.5'를 조만간 출시할 것이라고 밝혔다. 현 세대인 'GPT-4o'의 뒤를 잇는 마지막 '비추론 AI'로, 내부적으로는 '오라이언(Orion)'이라고 불렸다.

현재 챗GPT 이용자를 비롯한 오픈AI의 고객들은 'GPT-4o', 'o1', 'o3-미니', 'GPT-4' 등 모델들을 각자 선택해 활용하고 있다. 최신 모델은 'GPT-4'를 개선한 'GPT-4o'로, 'GPT-4'는 2023년 하반기, 'GPT-4o'는 2024년 상반기 출시됐다.

오픈AI는 'GPT-5'도 지난해 공개하려고 했으나, 예상보다 저조한 성과를 거둬 출시가 연기된 상태다. 이에 그간 연산 시간을 늘려 성능을 높인 'o'시리즈 추론 모델을 새롭게 내세웠다.

샘 알트먼 CEO는 "이후 공개될 'GPT-5'부터는 추론 모델인 'o'시리즈와 'GPT'를 통합하겠다"며 "모델과 제품라인이 복잡해졌음을 잘 알고 있고, 앞으로는 각 모델을 선택해 사용하기보다 그저 잘 작동하길 원한다"고 말했다.
    """

In [ ]:
user_query = f"""
당신의 목표는 주어진 기사를 요약하는 것입니다.
아래 주어진 기사 내용을 요약해주세요.
이전 시도의 요약과 피드백이 있다면, 이를 반영하여 개선된 요약을 작성하세요.

기사 내용 :
{input_article}
"""

In [ ]:
# LLM 모델 버전을 낮춰서 평가 결과가 fail이 나오게 만들기
summary = llm_call(user_query, model='gpt-3.5-turbo')

In [ ]:
# summary : gpt-3.5-turbo 모델로 user_query(기사내용)을 요약한 결과
summary

"오픈AI는 'GPT-4.5'를 곧 출시할 예정이며, 이를 통해 분산돼 있던 생성형 AI 모델을 통합할 계획이다. 'GPT-4'o'는 마지막 비추론 AI 모델로 'GPT-4.5'로 대체될 예정이며, 'GPT-5'는 지난해 예정보다 늦어져 연산 시간을 늘려 성능을 개선했다. 최근 공개된 'GPT-4o'와 이후 공개될 'GPT-5'를 추론 모델인 'o'시리즈와 통합할 예정이다."

In [ ]:
# final_evaluator_prompt : evaluator_prompt 하단 요약 결과 부분에 기사내용 요약한 summary 붙여넣음
final_evaluator_prompt = evaluator_prompt + summary

In [ ]:
print(final_evaluator_prompt)


다음 요약을 평가하십시오:

## 평가기준
1. 핵심 내용 포함 여부
   - 원문의 핵심 개념과 논리적 흐름이 유지되어야 합니다.
   - 불필요한 세부 사항은 줄이되, 핵심 정보가 누락되면 감점 요인입니다.
   - 단어 선택이 다소 달라도, 주요 개념과 의미가 유지되면 PASS 가능합니다.
   - 원문의 중요 개념 15% 이상이 빠졌다면 FAIL입니다.

2. 정확성 & 의미 전달
   - 요약이 원문의 의미를 왜곡하지 않고 정확하게 전달해야 합니다.
   - 숫자, 인명, 날짜 등 객관적 정보가 틀리면 FAIL입니다.
   - 문장이 다르게 표현되었더라도 원문의 의미를 유지하면 PASS 가능합니다.
   - 논리적 비약이 크거나 잘못된 해석이 포함되면 FAIL입니다.

3. 간결성 및 가독성
   - 문장이 과하게 길거나 반복적이면 감점 요인입니다.
   - 직역체 표현은 가독성을 해치지 않으면 허용 가능하지만, 지나치면 FAIL입니다.
   - 일부 단어의 표현 방식이 달라도 자연스럽다면 PASS 가능합니다.
   - 문장이 지나치게 어색해서 독해가 어렵다면 FAIL입니다.

4. 문법 및 표현
   - 맞춤법, 띄어쓰기 오류가 5개 이상이면 FAIL입니다.
   - 사소한 문법 실수는 감점 요인이나, 의미 전달에 영향을 주면 FAIL입니다.
   - 문장이 비문이거나 문맥상 어색한 표현이 많으면 FAIL입니다.

## 평가결과 응답예시
- 모든 기준이 충족되었으면 "평가결과 = PASS"를 출력하세요.
- 수정이 필요한 경우, 구체적인 문제점을 지적하고 반드시 개선 방향을 제시하세요.
- 중대한 오류가 있다면 "평가결과 = FAIL"을 출력하고, 반드시 주요 문제점을 설명하세요.

요약 결과 :
    오픈AI는 'GPT-4.5'를 곧 출시할 예정이며, 이를 통해 분산돼 있던 생성형 AI 모델을 통합할 계획이다. 'GPT-4'o'는 마지막 비추론 AI 모델로 'GPT-4.5'로 대체될 예정이며, 'GPT-5'는 지난해 예정보다 늦어져 연산 시간을

In [ ]:
# evaluation_result : 평가기준이랑 요약 결과를 LLM에 돌려서 평가 결과 및 피드백 출력
evaluation_result = llm_call(final_evaluator_prompt, model='gpt-4o')

In [ ]:
print(evaluation_result)

평가결과 = FAIL

주요 문제점:
1. 핵심 내용 포함 여부:
   - 요약본에서는 'GPT-4.5', 'GPT-4o', 'GPT-5'와 관련된 정보 및 계획이 혼재되어 있으며, 핵심 개념과 논리적 흐름이 명확하지 않습니다.
   - 'GPT-4o'라는 명칭은 혼란을 주고 있으며, 원문에 대한 오해를 불러일으킬 수 있습니다.
   - "추론 모델인 'o'시리즈"라는 부분도 혼동을 초래할 가능성이 큽니다.
   - 내용의 핵심을 제대로 전달하지 못하고 있습니다.

2. 정확성 & 의미 전달:
   - 'GPT-4o'와 'GPT-5'의 관계와 계획에 대한 설명이 명확하지 않고, 내용을 왜곡할 수 있습니다.
   - 정확한 정보 제공이 이루어지지 않았으며, 모델 간의 계획 통합이 직관적으로 이해되지 않습니다.

3. 간결성 및 가독성:
   - 문장이 모호하고 복잡하여 독자가 내용을 명확하게 이해하기 어렵습니다.
   - '오픈AI가 곧 출시할 계획', 'GPT-4o', '비추론 AI 모델' 등 불분명한 표현이 많아 가독성을 저해합니다.

4. 문법 및 표현:
   - 기본적인 문법 오류는 많지 않으나, 표현이 비문으로 보이는 부분이 있어 의미 전달에 방해가 될 수 있습니다.

개선 방향:
- 'GPT-4.5'와 'GPT-5'의 역할과 출시 계획을 명확하게 구분해 설명해야 합니다.
- 모델 간의 차이점과 함께 통합 계획을 구체적으로 설명하여 독자가 이해하기 쉽도록 작성하세요.
- 문장을 단순화하고 불필요한 반복을 제거하여 가독성을 높이세요.


In [ ]:
retries = 1
user_query += f'\n{retries}차 요약 결과: \n\n{summary}\n\n'

In [ ]:
user_query += f'{retries}차 요약 피드백: \n\n{evaluation_result}\n'

In [ ]:
print(user_query)


당신의 목표는 주어진 기사를 요약하는 것입니다.
아래 주어진 기사 내용을 요약해주세요.
이전 시도의 요약과 피드백이 있다면, 이를 반영하여 개선된 요약을 작성하세요.

기사 내용 :

오픈AI가 몇 주 안에 새로운 모델인 'GPT-4.5'를 출시하며 분산돼 있던 생성형 인공지능(AI) 모델을 통합키로 했다. 추론용 모델인 'o' 시리즈를 정리하고 비(非)추론 모델인 'GPT' 시리즈로 합칠 예정이다.

13일 업계에 따르면 샘 알트먼 오픈AI 최고경영자(CEO)는 지난 12일 자신의 X(옛 트위터)에 'GPT-4.5'를 조만간 출시할 것이라고 밝혔다. 현 세대인 'GPT-4o'의 뒤를 잇는 마지막 '비추론 AI'로, 내부적으로는 '오라이언(Orion)'이라고 불렸다.

현재 챗GPT 이용자를 비롯한 오픈AI의 고객들은 'GPT-4o', 'o1', 'o3-미니', 'GPT-4' 등 모델들을 각자 선택해 활용하고 있다. 최신 모델은 'GPT-4'를 개선한 'GPT-4o'로, 'GPT-4'는 2023년 하반기, 'GPT-4o'는 2024년 상반기 출시됐다.

오픈AI는 'GPT-5'도 지난해 공개하려고 했으나, 예상보다 저조한 성과를 거둬 출시가 연기된 상태다. 이에 그간 연산 시간을 늘려 성능을 높인 'o'시리즈 추론 모델을 새롭게 내세웠다.

샘 알트먼 CEO는 "이후 공개될 'GPT-5'부터는 추론 모델인 'o'시리즈와 'GPT'를 통합하겠다"며 "모델과 제품라인이 복잡해졌음을 잘 알고 있고, 앞으로는 각 모델을 선택해 사용하기보다 그저 잘 작동하길 원한다"고 말했다.
    

1차 요약 결과: 

오픈AI는 'GPT-4.5'를 곧 출시할 예정이며, 이를 통해 분산돼 있던 생성형 AI 모델을 통합할 계획이다. 'GPT-4'o'는 마지막 비추론 AI 모델로 'GPT-4.5'로 대체될 예정이며, 'GPT-5'는 지난해 예정보다 늦어져 연산 시간을 늘려 성능을 개선했다. 최근 공개된 'GPT-4o'와 이후 공개될 'GPT-5'를 추론 모델인 'o'시리즈와 통합할

피드백 5차까지 받아내기

1. summary 얻어내기
2. final_evaluator_prompt
3. evaluation_result
4. 평가 결과 = PASS / FAIL -> 프로그램 종료/ 계속 결정
5. user_query += 요약 결과, 피드백

In [ ]:
user_query = f"""
당신의 목표는 주어진 기사를 요약하는 것입니다.
아래 주어진 기사 내용을 요약해주세요.
이전 시도의 요약과 피드백이 있다면, 이를 반영하여 개선된 요약을 작성하세요.

기사 내용 :
{input_article}
"""

Langchain

In [ ]:
# LLM기반 애플리케이션(오케스트레이션,에이전트,RAG)을 만들기 위한 라이브러리 세트 설치 명령어
!pip install langchain-openai langgraph langchain-text-splitters python-dotenv pydantic langchain_community faiss-cpu retry langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.1/476.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.3 MB/s eta 0:00:00


In [ ]:
for i in range(5) :
  summary = llm_call(user_query, model='gpt-3.5-turbo')
  print(f"\n{i+1} 요약 결과 :\n", summary)

  final_evaluator_prompt = evaluator_prompt + summary

  evaluation_result = llm_call(final_evaluator_prompt, model='gpt-4o')
  print(f'\n{i+1} 평가 결과\n', evaluation_result)

  user_query += f'\n{i+1}차 요약 결과: \n\n{summary}\n\n'
  user_query += f'{i+1}차 요약 피드백: \n\n{evaluation_result}\n'

  if '평가결과 = PASS' in evaluation_result :
    print('\n 평가 통과 -> 프로그램 종료')
    break
    print('\n 평가 실패 -> 피드백 반영 후 재시도')


1 요약 결과 :
 원문의 내용을 잘 파악하여 개선된 요약을 작성해 주셔서 감사합니다. 추가로 개선 사항을 반영하여 주신 노력에 대해 감사드립니다. 요약의 논리적인 구조와 명확한 정보 전달로 인해 성공적인 결과를 얻을 수 있었습니다. 요약 작업을 이어가며 계속해서 논리적이고 명확한 내용을 전달하는 훌륭한 요약을 작성해주시기를 바랍니다. 제안드린 개선 사항을 참고하여 다음 요약 작업에도 도움이 되었으면 좋겠습니다. 계속된 노력에 감사드리며, 수고하셨습니다.

1 평가 결과
 평가결과 = FAIL

주요 문제점:
1. 핵심 내용 포함 여부 - 요약에서 원문의 특정 핵심 내용과 개념이 구체적으로 언급되지 않았습니다. 원문에서 전달하고자 하는 내용이나 주제를 구체적으로 다루지 않아서 불필요한 포괄적인 감사 표현으로 대체되었습니다.
2. 정확성 & 의미 전달 - 요약된 내용은 원문에서 전달하려고 하는 특정 정보나 데이터, 논리적 흐름을 정확히 재현하지 못하고 있습니다. 예를 들어, 원문의 특정 개념, 데이터, 혹은 예시가 전혀 없는 상태로 단순히 긍정적인 피드백 형태로만 존재합니다.
3. 간결성 및 가독성 - 요약이 다소 길고 반복적인 표현을 사용하고 있습니다. 간결하게 요약할 수 있는 부분이 많음에도 불구하고 비효율적인 반복으로 인해 긴 문장이 되었습니다.
4. 문법 및 표현 - 문장의 구조가 직역체로 되어 있어서, 가독성이 떨어지고 의미 전달이 불만족스럽습니다.

개선 방향:
- 실제 원문에서 강조된 핵심 개념과 논리적 흐름을 파악하여 요약에 반영하십시오.
- 감정적인 표현보다 특정 데이터를 포함하여 구체적이고 명확한 의미 전달에 집중하세요.
- 문장을 간결하게 유지하고 불필요한 반복을 줄여 가독성을 높이세요.
- 자연스럽고 문맥에 맞는 표현을 사용하여 문장을 구성하세요.

2 요약 결과 :
 개선된 요약에 대한 평가 결과는 'PASS'입니다. 새로운 내용을 요약할 때 논리적인 구조와 명확한 정보 전달을 유지하는 데 성공하셨습니다. 계속해서 논리적이고 명확한 내용

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
messages = [SystemMessage("You are a helpful assistant."),
            HumanMessage("안녕하세요! 저는 시오닝이라고 합니다"),
            AIMessage('안녕하세요. 시오닝님 어떤 도움이 필요하신가요?'),
            HumanMessage('제 이름을 아시나요?')]

In [ ]:
ai_message = model.invoke(messages)
print(ai_message.content)

네, 시오닝님이라고 말씀하셨습니다. 어떻게 도와드릴까요?


In [ ]:
# 왜 content를 넣은건지??
messages = [SystemMessage(content="You are a helpful assistant."),
            HumanMessage("안녕하세요! 저는 시오닝이라고 합니다"),
            AIMessage(content='안녕하세요. 시오닝님 어떤 도움이 필요하신가요?'),
            HumanMessage(content='제 이름을 아시나요?')]

In [ ]:
ai_message = model.invoke(messages)
print(ai_message.content)

네, 시오닝님이라고 말씀하셨습니다. 어떻게 도와드릴까요?


invoke를 사용할 수 있는 객체를 Runnable(실행가능한)이라고 함   

- invoke  
-> client.chat.completions.create() 이 역할을 함  
-> client는 OpenAI() 객체  # client = OpenAI()


<LangChain 방식>
```python
ai_message = model.invoke(messages)
print(ai_message.content)
```


<기존 방식>
```python
response = llm_call(prompt)
print(response)
#client.chat.completions.create(model=model,messages=[{"role": "user", "content": prompt}])
```





In [ ]:
for chunk in model.stream(messages):
  print(chunk.content)


네
,
 시
오
닝
님
이라고
 말씀
하
셨
습니다
.
 어떻게
 도
와
드
릴
까요
?





In [ ]:
# LLM 응답을 한번에 받지 않고 실시간 스트리밍으로 출력 -> 글자가 타자치듯 나오는 것 같은 효과
# flush : 실시간 출력되게 해줌 -> 안하면 출력 한번에 나옴
for chunk in model.stream(messages):
  print(chunk.content, end= "", flush=True)

네, 시오닝님이라고 말씀하셨습니다. 어떻게 도와드릴까요?

- model.stream(messages)  
-> LLM 응답을 조각(chunk) 단위로 순차 반환하는 generator
  - invoke() : 한번에 전체 응답
  - stream() : 생성되는 즉시 조금씩 응답

- for chunk in ...   
-> stream()이 생성한거 하나씩 꺼내기 위해 반복문 사용  

- chunk.content  
-> chunk는 AIMessageChunk 객체로 실제 텍스트는 chunk.content에 있음

- print(..., end="", flush=True)  
  - end="" : 줄바꿈 안함
  - flush=True : 출력 버퍼를 채워질때 까지 기다리는게 아니라 바로 비워서 지연없이 바로 화면에 출력

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template('다음 요리의 레시피를 생각해 주세요. 요리명: {dish}')

In [ ]:
prompt_value = prompt.invoke({'dish' : '라면'})
print(prompt_value.text)

다음 요리의 레시피를 생각해 주세요. 요리명: 라면


In [ ]:
ai_message = model.invoke(prompt_value)
print(ai_message.content)

라면은 간편하면서도 맛있는 요리로, 다양한 방법으로 조리할 수 있습니다. 아래는 기본적인 라면 레시피입니다.

### 기본 라면 레시피

#### 재료
- 라면 1봉지
- 물 550ml (약 2컵 반)
- 스프 (라면 봉지에 포함된 것)
- 선택 재료 (선택 사항)
  - 계란 1개
  - 대파 1대 (송송 썰기)
  - 김치 (적당량)
  - 채소 (시금치, 버섯, 애호박 등)
  - 고추 (매운맛을 원할 경우)
  - 치즈 (추가 시 부드러운 맛)

#### 조리 방법
1. **물 끓이기**: 냄비에 물 550ml를 붓고 중불에서 끓입니다.
  
2. **면 넣기**: 물이 끓기 시작하면 라면 면을 넣고 3-4분간 끓입니다. 면이 부드러워질 때까지 끓여주세요.

3. **스프 추가**: 면이 익으면 라면 봉지에 포함된 스프를 넣고 잘 저어줍니다. 이때 선택 재료(예: 대파, 채소 등)를 추가할 수 있습니다.

4. **계란 추가 (선택 사항)**: 계란을 넣고 싶다면, 면이 거의 다 익었을 때 계란을 깨뜨려 넣고 저어줍니다. 계란이 익을 때까지 1-2분 더 끓입니다.

5. **마무리**: 모든 재료가 잘 섞이고 익으면 불을 끄고, 그릇에 담습니다. 원한다면 김치나 고추, 치즈 등을 올려서 서빙합니다.

6. **즐기기**: 뜨거운 라면을 맛있게 즐기세요!

### 팁
- 라면의 국물 맛을 더 깊게 하고 싶다면, 멸치나 다시마로 육수를 내서 물 대신 사용해 보세요.
- 다양한 재료를 추가하여 나만의 라면을 만들어 보세요! 

맛있게 드세요!


- PromptTemplate.from_template : '단일 문자열 프롬프트'용  
  - 출력 : 문자열
  - 역할(role) 개념 X
  - 단일 입력 -> 단일 텍스트

- ChatPromptTemplate.from_messages : '대화(Message) 기반 프롬프트'용  
  - 출력 : 메시지 리스트
  - 역할 구분 : system/user/assistant
  - Chat 모델 전용

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '사용자가 입력한 요리의 레시피를 생각해 주세요.'),
    ('human', '{dish}')])

In [ ]:
prompt_value = prompt.invoke({'dish' : '라면'})
print(prompt_value)

messages=[SystemMessage(content='사용자가 입력한 요리의 레시피를 생각해 주세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='라면', additional_kwargs={}, response_metadata={})]


In [ ]:
prompt_value.messages[0].content

'사용자가 입력한 요리의 레시피를 생각해 주세요.'

In [ ]:
# 대화 히스토리(과거 대화 이력)를 포함할 수 있는 chatprompttemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant'), # system 메시지
    MessagesPlaceholder('chat_history', optional=True), # 이전 대화
    ('human', '{input}')]) # 이번 사용자 입력



```
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant'), # system 메시지
    MessagesPlaceholder('chat_history', optional=True), # 이전 대화
    ('human', '{input}')])
```
- MessagesPlaceholder('chat_history', optional=True)  
-> 이 자리에 '이전 대화 메시지들'을 통째로 끼워 넣겠다

- optional=True : 이전 대화가 없어도 에러 안남
- ('human', '{input}') : 이번 턴의 사용자 질문
  - invoke할때 input 있음

In [ ]:
prompt_value = prompt.invoke({'chat_history':[
    HumanMessage('안녕하세요! 저는 시오닝이라고 합니다.'),
    AIMessage('안녕하세요, 시오닝님 어떻게 도와드릴까요')], 'input':'제이름을 아시나요?'})
print(prompt_value)

messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요! 저는 시오닝이라고 합니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 시오닝님 어떻게 도와드릴까요', additional_kwargs={}, response_metadata={}), HumanMessage(content='제이름을 아시나요?', additional_kwargs={}, response_metadata={})]


In [ ]:
ai_message = model.invoke(prompt_value)
print(ai_message)

content='네, 시오닝님이라고 말씀하셨습니다! 어떻게 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 55, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ee69c2ef48', 'id': 'chatcmpl-CndNmUMhN20FDV0GN0P21uFyIwM3X', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019b2a8d-ccf2-7a43-8507-1a3cefd3d4ae-0' usage_metadata={'input_tokens': 55, 'output_tokens': 19, 'total_tokens': 74, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [ ]:
# LLM의 출력 결과를 str(문자열)로 형식 지정
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

parser  
-> LLM이 만든 응답을 프로그램이 다루기 쉬운 형태로 변환하는 변환기

In [ ]:
ai_message = output_parser.invoke(ai_message)

In [ ]:
# content만 나옴 -> StrOuputParser의 역할
print(ai_message)

네, 시오닝님이라고 말씀하셨습니다! 어떻게 도와드릴까요?


In [ ]:
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# 프롬프트 -> LLM -> 출력 파서를 파이프라인 하나로 연결한 LangChain
# | : 파이프 연산자, 왼쪽 출력 -> 오른쪽 입력
chain = prompt | model | output_parser

```
chain = prompt | model | output_parser
```
=> 아래 세 단계를 한 줄로 묶은 것임
1. prompt: 입력값을 받아 프롬프트를 만든다
2. model: 프롬프트를 LLM에 보내 응답을 생성한다
3. output_parser: 응답을 원하는 형태로 변환한다



In [ ]:
chain.invoke({'input':'태양의 공전 속도는?'})

'태양은 은하수의 중심을 공전하고 있으며, 그 속도는 약 230킬로미터/초(약 828,000킬로미터/시간)입니다. 태양이 은하수 중심을 한 바퀴 도는 데는 약 225-250백만 년이 걸립니다. 이 과정을 "은하년"이라고 부릅니다.'

In [ ]:
chain.invoke({'chat_history':[
    HumanMessage('안녕하세요! 저는 시오닝이라고 합니다.'),
    AIMessage('안녕하세요, 시오닝님 어떻게 도와드릴까요')], 'input':'제이름을 아시나요?'})

'네, 시오닝님이라고 말씀하셨습니다. 어떻게 도와드릴까요?'

In [ ]:
chain.invoke({'input':'태양의 공전 속도는?'})

In [ ]:
# 한국어 단어 -> 영어 번역하는 체인 만들기

In [ ]:
# 번역용 프롬프트 템플릿 생성
# {korean_word} 자리에 나중에 값이 들어감
prompt1 = ChatPromptTemplate.from_template('translates {korean_word} to English') # 한국어 단어를 영어로 번역하라는 지시임

# 사전(옥스포드) 스타일로 설명하는 프롬프트 템플릿 생성
# {english_word} 자리에 값이 들어감
prompt2 = ChatPromptTemplate.from_template('explain {english_word} using oxford dictionary to me in Korean.') # 영단어를 옥스포드 사전 스타일로 설명하되, 한국어로 다시 설명해라

In [ ]:
# 사용할 LLM(채팅모델) 객체 생성
# llm : 프롬프트를 입력받아 답변을 생성하는 실행 엔진
llm = ChatOpenAI(model='gpt-3.5-turbo-0125')

In [ ]:
# 파이프라인(체인) 구성
# prompt1 -> llm -> StrOutputParser() 순서로 연결됨
chain1 = prompt1 | llm | StrOutputParser()



```
chain1 = prompt1 | llm | StrOutputParser()
```
- prompt1 : {'korean_word':'...'}를 받아 최종 메시지 프롬프트 생성
- llm : 그 프롬프트를 OpenAI 모델에 보내 답변(AIMessage)를 받음
- StrOutputParser : AIMessage에서 텍스트(content)만 뽑아 최종 결과를 str으로 만듦


In [ ]:
chain1.invoke({'korean_word' : '내일'})

'Tomorrow'

In [ ]:
# chain1의 출력 결과를 prompt2의 {english_word} 변수에 넣어서 다음 LLM 작업 수행
# 번역 결과 -> 설명 입력으로 연결
chain2 = {'english_word':chain1} | prompt2 | llm | StrOutputParser()

In [ ]:
chain2.invoke({'korean_word' : '내일'})

'According to Oxford Dictionary, the word "Tomorrow" in Korean is "내일," which refers to the day that follows today.'

prompt2 내용을 한글로 나오게 프롬프트에 추가 요청 적어보기


In [ ]:
prompt = ChatPromptTemplate.from_template(
    '지구과학에서 {topic}에 대해 간단히 설명하시오')

In [ ]:
output_parser = StrOutputParser()
llm = ChatOpenAI(model='gpt-4o')
chain=prompt | llm | output_parser

In [ ]:
chain.invoke({'topic':'지구 자전'})

'지구 자전은 지구가 자신의 축을 중심으로 회전하는 운동을 말합니다. 지구는 약 23.5도 기울어진 상태로 자전하며, 이로 인해 낮과 밤이 생기고, 계절의 변화에도 영향을 미칩니다. 지구는 서쪽에서 동쪽으로, 즉 반시계 방향으로 자전하기 때문에 태양은 동쪽에서 떠서 서쪽으로 지는 것처럼 보입니다. 한 번 자전하는 데 걸리는 시간은 약 24시간으로, 이를 통해 하루가 완성됩니다. 자전의 속도는 적도 부근에서 가장 빠르며, 극지방으로 갈수록 느려집니다.'

In [ ]:
topics = ['지구 공전', '대륙 이동', '화산 활동']
results = chain.batch([{'topic':t} for t in topics])

In [ ]:
for i in results:
  print(i)
  print('+'*50)


지구 공전은 지구가 태양을 중심으로 궤도를 그리며 도는 운동을 의미합니다. 지구는 태양 주위를 타원 궤도를 그리며 약 365.25일(1년)에 한 바퀴를 돕니다. 이 공전 운동 때문에 지구에는 각기 다른 계절이 발생합니다. 공전 궤도는 약간 찌그러진 타원 형태이며, 지구의 자전축은 약 23.5도 기울어져 있어 계절 변화에 중요한 역할을 합니다. 지구가 태양 주위를 공전하면서 자전축의 기울기와 태양의 위치 변화에 따라 일조량이 달라져 봄, 여름, 가을, 겨울 같은 계절이 생기게 됩니다.
++++++++++++++++++++++++++++++++++++++++++++++++++
대륙 이동은 지구의 대륙들이 지질학적 시간 척도에 따라 서로 다른 위치로 이동하는 과정을 의미합니다. 이 이론은 20세기 초 독일의 기상학자 알프레드 베게너(Alfred Wegener)에 의해 처음 제안되었습니다. 그는 모든 대륙이 한때 "판게아"라고 불리는 하나의 거대한 초대륙으로 모여 있었으며, 이후 여러 시기에 걸쳐 현재의 위치로 이동했다고 주장했습니다.

대륙 이동의 주된 원인은 지구의 맨틀에서 발생하는 열 대류와 판 구조론(plate tectonics)에 의해 설명됩니다. 지구의 외곽은 여러 개의 커다란 판으로 나뉘며, 이 판들은 맨틀 위에서 천천히 움직입니다. 이 과정에서 충돌, 분리, 미끄러짐 등의 상호작용이 발생하며, 이는 지진, 화산 활동, 산맥 형성 등의 지질학적 현상을 유발합니다.

대륙 이동 이론은 초기에는 많은 과학자들에게 비판을 받았으나, 해저 지형의 발견과 판 구조론의 발전을 통해 그 근거가 강화되었습니다. 이로 인해 현재는 지질학의 중요한 이론 중 하나로 인정받고 있습니다.
++++++++++++++++++++++++++++++++++++++++++++++++++
화산 활동은 지구 내부의 마그마가 지각을 뚫고 표면으로 분출하는 과정을 말합니다. 이 과정은 여러 단계를 거치며 다양한 형태로 나타날 수 있습니다. 기본적으로, 지구 내부 깊숙한 곳에서 높은 온도로 인해

In [ ]:
stream = chain.stream({'topic':'지진'})

In [ ]:
stream

<generator object RunnableSequence.stream at 0x797273aa34c0>

In [ ]:
for i in stream:
  print(i)


지
진
은
 지
구
 표
면
에서
 발생
하는
 갑
작
스
러운
 진
동
 현
상
으로
,
 주
로
 지
각
(
지
구
의
 가장
 바
깥
층
)의
 판
들이
 움직
이
면서
 발생
합니다
.
 이러한
 판
의
 움직
임
은
 대
개
 판
 사이
의
 축
적
된
 응
력이
 갑
작
스럽
게
 방
출
될
 때
 발생
하며
,
 그
 결과
로
 지
진
파
가
 생성
됩니다
.
 이런
 지
진
파
는
 지
구
 내부
를
 통
과
하며
 지
표
면
에
 흔
들
림
을
 일
으
키
게
 됩니다
.


지
진
은
 규모
와
 깊
이에
 따라
 다양한
 영향을
 미
칠
 수
 있습니다
.
 작은
 규모
의
 지
진
은
 사람이
 느
끼
지
 못
할
 수도
 있지만
,
 큰
 규모
의
 지
진
은
 건
물
의
 파
괴
와
 인
명
 피해
를
 초
래
할
 수
 있습니다
.
 지
진
의
 강
도
는
 일반
적으로
 릭
터
 규모
라는
 척
도를
 사용
해
 측
정
됩니다
.
 진
앙
지는
 지
진
이
 시작
된
 지
점
이고
,
 진
원
지는
 지
구
 내부
의
 실제
 발생
 지
점
입니다
.
 지
진
은
 자연
재
해
 중
 하나
로
,
 특히
 지
진
대
에
 위치
한
 지역
에서는
 건
축
과
 안전
 대비
가
 매우
 중요
합니다
.





In [ ]:
for i in stream:
  print(i, end="")

지진은 지구 표면에서 발생하는 자연 현상으로, 지구 내부에서 축적된 에너지가 갑작스럽게 방출되면서 지진파를 발생시키는 과정을 의미합니다. 이러한 에너지는 주로 지각의 판들이 서로 부딪히거나 이동하면서 축적됩니다. 

지진의 발생 지점은 '진원'이라고 하며, 진원의 바로 위 지표면 지점을 '진앙'이라고 합니다. 지진의 규모는 일반적으로 리히터 규모나 모멘트 규모를 사용하여 측정되며, 사람들이 느끼는 진동의 강도는 '진도'라는 단위로 표현됩니다.

지진은 다양한 자연적, 인위적 요인에 의해 발생할 수 있으며, 그 결과로 건물 붕괴, 산사태, 쓰나미 등 심각한 피해를 야기할 수 있습니다. 따라서 지진을 모니터링하고 대비하는 것이 매우 중요합니다.

In [ ]:
# Few-shot : 소수 예시 학습
# FewShotChatMessagePromptTemplate : 예시 여러개 반복 삽입
from langchain_core.prompts import FewShotChatMessagePromptTemplate

In [ ]:
# Few-shot 학습을 위한 '예시 데이터', '예시 데이터에 대한 예시 답변'
examples = [
    {"input": "지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가요?", "output": "질소입니다."},
    {"input": "광합성에 필요한 주요 요소들은 무엇인가요?", "output": "빛, 이산화탄소, 물입니다."},
]

In [ ]:
example_prompt = ChatPromptTemplate.from_messages(
    [
        ('human','{input}'),
        ('ai', '{output}')
    ]
)

In [ ]:
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

In [ ]:
final_prompt = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 과학과 수학에 대해 잘 아는 교육자다.'),
        few_shot_prompt,
        ('human','{input}')
    ]
)

In [ ]:
model = ChatOpenAI(model='gpt-4o', temperature=0)

In [ ]:
chain3 = final_prompt | model | StrOutputParser()

In [ ]:
chain3.invoke({'input':'빛의 속도'})

'빛의 속도는 진공에서 약 299,792,458 미터/초입니다. 일반적으로 약 3.00 x 10^8 m/s로 표현됩니다.'

In [ ]:
# 모델 설정 시 max_tokens 사용
model2 = ChatOpenAI(model='gpt-4o', max_tokens=20)

chain4 = final_prompt | model2 | StrOutputParser()

chain4.invoke({'input':'빛의 속도'})

'빛의 속도는 진공에서 약 299,792,458 미터/초입니다'

In [ ]:
llm_ =

### CSV 파서


In [ ]:
# 출력 파서(Output Parser) 중 콤마(,)로 구분된 리스트 형태를 다루는 파
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser() # 모델이 낸 텍스트 결과를 ['항목1','항목2'..] 같은 파이썬 리스트로 변환해주는 객체 생성

# 모델한테 어떻게 출력하라고 말해야 하는지에 대한 프롬프트용 안내문을 자동으로 만들어줌
format_instructions = output_parser.get_format_instructions()

print(format_instructions) # 실제로 모델에게 보여줄 출력 형식 지시문 출력

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [ ]:
prompt = PromptTemplate(
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={'format_instructions':format_instructions}
)

In [ ]:
llm = ChatOpenAI(model='gpt-4o', temperature=0)

chain = prompt | llm | output_parser

chain.invoke({'subject':'popular Korean cuisine'})

['Kimchi', 'Bibimbap', 'Bulgogi', 'Tteokbokki', 'Japchae']

### JSON 파서

Pydantic의 Field는 BaseModel의 필드를 더 자세하게 정의하고 검증하기 위한 도구.  
단순히 타입을 지정하는 것을 넘어서, 데이터의 범위, 형식, 기본값 등을 세밀하게 제어할 수 있게 해줌

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field #pydantic : 데이터 타입 강제하기 위해 가져옴

# 자료 구조 정의(pydanic)
class CuisineRecipe(BaseModel): # BaseModel이 name, recipe 넣게 함
  name: str = Field(description='name of a cuisine')
  recipe : str = Field(description='recipe to cook the cuisine')

# 출력 파서 정의
output_parser = JsonOutputParser(pydantic_object=CuisineRecipe)

format_instructions = output_parser.get_format_instructions()

print(format_instructions)

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [ ]:
prompt = PromptTemplate(template='Answer the user query. \n {format_instructions}\n{query}\n',
                        input_variables=['query'],
                        partial_variables={'format_instructions':format_instructions})

In [ ]:
chain = prompt | llm | output_parser

chain.invoke({'query':'Let me knew how to cook Bibimbap'})

{'name': 'Bibimbap',
 'recipe': '1. Cook rice according to package instructions. 2. Prepare vegetables: julienne carrots, slice cucumbers, and blanch spinach. 3. Cook bean sprouts and season with sesame oil and salt. 4. Cook beef or tofu with soy sauce, garlic, and sugar. 5. Fry an egg sunny-side up. 6. Assemble: place rice in a bowl, arrange vegetables and protein on top, and add the egg. 7. Serve with gochujang (Korean chili paste) and mix everything before eating.'}

In [ ]:
CR_IO = CuisineRecipe(name='곤', recipe='없음')

In [ ]:
CR_IO.recipe